In [21]:
import sys

sys.path.append("../")

In [22]:
import json
import pandas as pd
from datasets import load_dataset

In [23]:
swebench = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

In [24]:
swebench[30]

{'repo': 'django/django',
 'instance_id': 'django__django-12125',
 'base_commit': '89d41cba392b759732ba9f1db4ff29ed47da6a56',
 'patch': 'diff --git a/django/db/migrations/serializer.py b/django/db/migrations/serializer.py\n--- a/django/db/migrations/serializer.py\n+++ b/django/db/migrations/serializer.py\n@@ -269,7 +269,7 @@ def serialize(self):\n             if module == builtins.__name__:\n                 return self.value.__name__, set()\n             else:\n-                return "%s.%s" % (module, self.value.__name__), {"import %s" % module}\n+                return "%s.%s" % (module, self.value.__qualname__), {"import %s" % module}\n \n \n class UUIDSerializer(BaseSerializer):\n',
 'test_patch': 'diff --git a/tests/migrations/test_writer.py b/tests/migrations/test_writer.py\n--- a/tests/migrations/test_writer.py\n+++ b/tests/migrations/test_writer.py\n@@ -26,6 +26,11 @@\n from .models import FoodManager, FoodQuerySet\n \n \n+class DeconstructibleInstances:\n+    def deconstruct

In [25]:
f_test = None

for d in swebench:
    if d["instance_id"]=="matplotlib__matplotlib-25442":
        f_test = d
        break

In [26]:
print(f_test["patch"])

diff --git a/lib/matplotlib/offsetbox.py b/lib/matplotlib/offsetbox.py
--- a/lib/matplotlib/offsetbox.py
+++ b/lib/matplotlib/offsetbox.py
@@ -1500,16 +1500,23 @@ def __init__(self, ref_artist, use_blit=False):
             ref_artist.set_picker(True)
         self.got_artist = False
         self._use_blit = use_blit and self.canvas.supports_blit
-        self.cids = [
-            self.canvas.callbacks._connect_picklable(
-                'pick_event', self.on_pick),
-            self.canvas.callbacks._connect_picklable(
-                'button_release_event', self.on_release),
+        callbacks = ref_artist.figure._canvas_callbacks
+        self._disconnectors = [
+            functools.partial(
+                callbacks.disconnect, callbacks._connect_picklable(name, func))
+            for name, func in [
+                ("pick_event", self.on_pick),
+                ("button_release_event", self.on_release),
+                ("motion_notify_event", self.on_motion),
+          

In [27]:
file_path = "../results/rq1-1/blagent_gpt-oss.json"

In [28]:
with open(file_path, "r") as f:
    data = json.load(f)

In [29]:
from pprint import pprint

In [30]:
swebench[145]

{'repo': 'mwaskom/seaborn',
 'instance_id': 'mwaskom__seaborn-3190',
 'base_commit': '4a9e54962a29c12a8b103d75f838e0e795a6974d',
 'patch': 'diff --git a/seaborn/_core/scales.py b/seaborn/_core/scales.py\n--- a/seaborn/_core/scales.py\n+++ b/seaborn/_core/scales.py\n@@ -346,7 +346,7 @@ def _setup(\n                 vmin, vmax = data.min(), data.max()\n             else:\n                 vmin, vmax = new.norm\n-            vmin, vmax = axis.convert_units((vmin, vmax))\n+            vmin, vmax = map(float, axis.convert_units((vmin, vmax)))\n             a = forward(vmin)\n             b = forward(vmax) - forward(vmin)\n \n',
 'test_patch': 'diff --git a/tests/_core/test_scales.py b/tests/_core/test_scales.py\n--- a/tests/_core/test_scales.py\n+++ b/tests/_core/test_scales.py\n@@ -90,6 +90,12 @@ def test_interval_with_range_norm_and_transform(self, x):\n         s = Continuous((2, 3), (10, 100), "log")._setup(x, IntervalProperty())\n         assert_array_equal(s(x), [1, 2, 3])\n \n+    de

In [31]:
data[202]

{'swe_data_index': 202,
 'problem_statement': "regression in input validation of clustering metrics\n```python\r\nfrom sklearn.metrics.cluster import mutual_info_score\r\nimport numpy as np\r\n\r\nx = np.random.choice(['a', 'b'], size=20).astype(object)\r\nmutual_info_score(x, x)\r\n```\r\nValueError: could not convert string to float: 'b'\r\n\r\nwhile\r\n```python\r\nx = np.random.choice(['a', 'b'], size=20)\r\nmutual_info_score(x, x)\r\n```\r\nworks with a warning?\r\n\r\nthis worked in 0.21.1 without a warning (as I think it should)\r\n\r\n\r\nEdit by @ogrisel: I removed the `.astype(object)` in the second code snippet.\n",
 'augmented_query': ['mutual_info_score sklearn.metrics.cluster input validation dtype object conversion to float error regression; check_array or label validation in mutual_info_score implementation; look for changes in _contingency_matrix or _validate_input after version 0.21.1 causing ValueError for string labels; relevant modules: sklearn.metrics.cluster._sup

In [32]:
print(data[119]["problem_statement"])

Class methods from nested classes cannot be used as Field.default.
Description
	 
		(last modified by Mariusz Felisiak)
	 
Given the following model:
 
class Profile(models.Model):
	class Capability(models.TextChoices):
		BASIC = ("BASIC", "Basic")
		PROFESSIONAL = ("PROFESSIONAL", "Professional")
		
		@classmethod
		def default(cls) -> list[str]:
			return [cls.BASIC]
	capabilities = ArrayField(
		models.CharField(choices=Capability.choices, max_length=30, blank=True),
		null=True,
		default=Capability.default
	)
The resulting migration contained the following:
 # ...
	 migrations.AddField(
		 model_name='profile',
		 name='capabilities',
		 field=django.contrib.postgres.fields.ArrayField(base_field=models.CharField(blank=True, choices=[('BASIC', 'Basic'), ('PROFESSIONAL', 'Professional')], max_length=30), default=appname.models.Capability.default, null=True, size=None),
	 ),
 # ...
As you can see, migrations.AddField is passed as argument "default" a wrong value "appname.models.Capab

In [33]:
print(data[119]["augmented_query"][0])

Django migrations default callable resolution for nested classes – issue in `Field.deconstruct` / `ArrayField.deconstruct` generating wrong import path (`appname.models.Capability.default` instead of `appname.models.Profile.Capability.default`). Search for handling of `default.__qualname__` in `django.db.migrations.writer` or `django.db.migrations.state` when serializing callables for `ArrayField`. Look for bugs in `django.contrib.postgres.fields.ArrayField` default handling and migration writer logic for nested `TextChoices` classes. Probable cause: incorrect qualification of nested class method during migration serialization.


In [34]:
only_structural = []
only_behavioral = []

for entry in data:
    if entry["is_patch_in_top10_t0"] and not entry["is_patch_in_top10_t1"]:
        only_structural.append(entry)
    if entry["is_patch_in_top10_t1"] and not entry["is_patch_in_top10_t0"]:
        only_behavioral.append(entry)

In [35]:
len(only_structural), len(only_behavioral)

(17, 7)

In [36]:
index = 6
d = only_structural[index]
print("Actual Patch:", d["patch_file"])
print(d["problem_statement"])

Actual Patch: seaborn/_core/scales.py
Color mapping fails with boolean data
```python
so.Plot(["a", "b"], [1, 2], color=[True, False]).add(so.Bar())
```
```python-traceback
---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
...
File ~/code/seaborn/seaborn/_core/plot.py:841, in Plot._plot(self, pyplot)
    838 plotter._compute_stats(self, layers)
    840 # Process scale spec for semantic variables and coordinates computed by stat
--> 841 plotter._setup_scales(self, common, layers)
    843 # TODO Remove these after updating other methods
    844 # ---- Maybe have debug= param that attaches these when True?
    845 plotter._data = common

File ~/code/seaborn/seaborn/_core/plot.py:1252, in Plotter._setup_scales(self, p, common, layers, variables)
   1250     self._scales[var] = Scale._identity()
   1251 else:
-> 1252     self._scales[var] = scale._setup(var_df[var], prop)
   1254 # Everythi

In [37]:
d

{'swe_data_index': 145,
 'problem_statement': 'Color mapping fails with boolean data\n```python\r\nso.Plot(["a", "b"], [1, 2], color=[True, False]).add(so.Bar())\r\n```\r\n```python-traceback\r\n---------------------------------------------------------------------------\r\nTypeError                                 Traceback (most recent call last)\r\n...\r\nFile ~/code/seaborn/seaborn/_core/plot.py:841, in Plot._plot(self, pyplot)\r\n    838 plotter._compute_stats(self, layers)\r\n    840 # Process scale spec for semantic variables and coordinates computed by stat\r\n--> 841 plotter._setup_scales(self, common, layers)\r\n    843 # TODO Remove these after updating other methods\r\n    844 # ---- Maybe have debug= param that attaches these when True?\r\n    845 plotter._data = common\r\n\r\nFile ~/code/seaborn/seaborn/_core/plot.py:1252, in Plotter._setup_scales(self, p, common, layers, variables)\r\n   1250     self._scales[var] = Scale._identity()\r\n   1251 else:\r\n-> 1252     self._

In [38]:
pprint(d["augmented_query"][0])

('seaborn._core.scales ContinuousBase._setup boolean subtraction TypeError; '
 'handling of `color` argument with bool array in Plot._setup_scales; '
 'conversion of boolean data to numeric or categorical scale in '
 'seaborn/_core/scales.py; probable issue in scale type detection for boolean '
 'semantic variables.')


In [39]:
print(d["augmented_query"][1])

Seaborn Plot raises a `TypeError` when a boolean list is passed to the `color` argument (e.g., `Plot(["a","b"], [1,2], color=[True, False]).add(Bar())`). Expected: boolean values should be treated as categorical colors or produce a clear error; actual: the `ContinuousBase._setup` scale attempts numeric subtraction on booleans, causing `numpy boolean subtract` error. Trigger: calling `Plot` with `color` set to a boolean array. Likely module: `seaborn._core.scales` (continuous scale handling) misclassifying boolean data as continuous. Possible cause: missing type check/fallback for boolean inputs in scale setup.


In [40]:
only_behavioral

[{'swe_data_index': 107,
  'problem_statement': 'ModelForm fields with callable defaults don\'t correctly propagate default values\nDescription\n\t\nWhen creating an object via the admin, if an inline contains an ArrayField in error, the validation will be bypassed (and the inline dismissed) if we submit the form a second time (without modification).\ngo to /admin/my_app/thing/add/\ntype anything in plop\nsubmit -> it shows an error on the inline\nsubmit again -> no errors, plop become unfilled\n# models.py\nclass Thing(models.Model):\n\tpass\nclass RelatedModel(models.Model):\n\tthing = models.ForeignKey(Thing, on_delete=models.CASCADE)\n\tplop = ArrayField(\n\t\tmodels.CharField(max_length=42),\n\t\tdefault=list,\n\t)\n# admin.py\nclass RelatedModelForm(forms.ModelForm):\n\tdef clean(self):\n\t\traise ValidationError("whatever")\nclass RelatedModelInline(admin.TabularInline):\n\tform = RelatedModelForm\n\tmodel = RelatedModel\n\textra = 1\n@admin.register(Thing)\nclass ThingAdmin(adm

In [41]:
from blagent.chroma.chroma_store import get_splitter, load_or_build_vector_store

/local/home/amamun/envs/ragfix/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [42]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings

DATA_DIR = Path("./swebench_data")
REPO_ROOT = Path("./repo_data")
CHROMA_ROOT = Path("chroma_data_prune")

EMBED_MODEL = "nomic-ai/nomic-embed-text-v1"
LLM_MODEL = "gpt-oss:120b"
EMBED_MODEL_KWARGS = {"device": "cuda", "trust_remote_code": True}
ENCODE_KWARGS = {"normalize_embeddings": False}
INCLUDE_PATH_IN_CHUNK = True


def make_embed_model(
    model_name: str = EMBED_MODEL,
    model_kwargs: dict = EMBED_MODEL_KWARGS,
    encode_kwargs: dict = ENCODE_KWARGS,
):
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs,
        show_progress=True,
    )


In [43]:
embed_model = make_embed_model()

<All keys matched successfully>


In [44]:
splitter = get_splitter("code")

[_get_splitter] using code splitter


In [45]:
swe_index = 81
swebench_entry = swebench[swe_index]

In [46]:
data_point = [d for d in data if d["swe_data_index"] == swe_index][0]
data_point

{'swe_data_index': 81,
 'problem_statement': "Wrong URL generated by get_admin_url for readonly field in custom Admin Site\nDescription\n\t\nWhen a model containing a ForeignKey field is viewed (or edited) in a custom Admin Site, and that ForeignKey field is listed in readonly_fields, the url generated for the link is /admin/... instead of /custom-admin/....\nThis appears to be caused by the following line in django.contrib.admin.helpers get_admin_url:\nurl = reverse(url_name, args=[quote(remote_obj.pk)])\nOther parts of the admin use the current_app keyword parameter to identify the correct current name of the Admin Site. (See django.contrib.admin.options.ModelAdmin response_add as just one example)\nI have been able to correct this specific issue by replacing the above line with:\nurl = reverse(\n\turl_name,\n\targs=[quote(remote_obj.pk)],\n\tcurrent_app=self.model_admin.admin_site.name\n)\nHowever, I don't know if there are any side effects and I have not yet run the full suite of t

In [47]:
vector_store = load_or_build_vector_store(
        swebench_entry,
        swe_index,
        embed_model,
        splitter,
        data_dir=DATA_DIR,
        chroma_root=CHROMA_ROOT,
        repo_root=REPO_ROOT,
        include_path_in_chunk=INCLUDE_PATH_IN_CHUNK,
    )

Cloning repository from https://github.com/django/django.git to repo_data/c11b671b-ca7b-4453-b883-8e351795e11c/django...


Cloning into 'repo_data/c11b671b-ca7b-4453-b883-8e351795e11c/django'...
fetch-pack: unexpected disconnect while reading sideband packet


KeyboardInterrupt: 

In [ ]:
def get_retriever_from_store(store, k: int = 1000):
    return store.as_retriever(search_kwargs={"k": k})

In [ ]:
retriever = get_retriever_from_store(vector_store)

problem_statement = swebench_entry.get("problem_statement", "")
patch = swebench_entry.get("patch", "")
augmented_query = data_point.get("augmented_query")[0]

retrieved_docs = retriever.get_relevant_documents(augmented_query)
retrieved_paths = []
paths = set()
for i, d in enumerate(retrieved_docs):
    paths.add(d.metadata.get("relative_path", ""))
    if d.metadata.get("relative_path", "") == data_point["patch_file"]:
        print(f"Found the actual patch in retrieved docs! Index: {i}")
        # print(d)
    # print(d)
    # retrieved_paths.append(d.metadata.get("relative_path", ""))
    # print("=="*20)

Batches: 100%|██████████| 1/1 [00:00<00:00, 131.25it/s]

Found the actual patch in retrieved docs! Index: 9
Found the actual patch in retrieved docs! Index: 38
Found the actual patch in retrieved docs! Index: 104
Found the actual patch in retrieved docs! Index: 144
Found the actual patch in retrieved docs! Index: 169
Found the actual patch in retrieved docs! Index: 194
Found the actual patch in retrieved docs! Index: 199
Found the actual patch in retrieved docs! Index: 229
Found the actual patch in retrieved docs! Index: 269
Found the actual patch in retrieved docs! Index: 364
Found the actual patch in retrieved docs! Index: 439
Found the actual patch in retrieved docs! Index: 467
Found the actual patch in retrieved docs! Index: 608
Found the actual patch in retrieved docs! Index: 671
Found the actual patch in retrieved docs! Index: 676
Found the actual patch in retrieved docs! Index: 719
Found the actual patch in retrieved docs! Index: 810


In [ ]:
paths

{'django/__init__.py',
 'django/apps/config.py',
 'django/apps/registry.py',
 'django/conf/__init__.py',
 'django/conf/global_settings.py',
 'django/conf/urls/__init__.py',
 'django/conf/urls/static.py',
 'django/contrib/admin/__init__.py',
 'django/contrib/admin/actions.py',
 'django/contrib/admin/apps.py',
 'django/contrib/admin/checks.py',
 'django/contrib/admin/decorators.py',
 'django/contrib/admin/exceptions.py',
 'django/contrib/admin/filters.py',
 'django/contrib/admin/forms.py',
 'django/contrib/admin/helpers.py',
 'django/contrib/admin/migrations/0001_initial.py',
 'django/contrib/admin/models.py',
 'django/contrib/admin/options.py',
 'django/contrib/admin/sites.py',
 'django/contrib/admin/templatetags/admin_list.py',
 'django/contrib/admin/templatetags/admin_modify.py',
 'django/contrib/admin/templatetags/admin_urls.py',
 'django/contrib/admin/templatetags/base.py',
 'django/contrib/admin/templatetags/log.py',
 'django/contrib/admin/utils.py',
 'django/contrib/admin/views/au